In [ ]:
## -- System dependencies --
import sys, os, gc
import torch

## -- Device-Agnostic (GPU/CPU) --
def get_system_info():
    if torch.cuda.is_available():
        return f"GPU: {torch.cuda.get_device_name(0)}"
    else:
        return f"CPU: {os.cpu_count()}"

get_system_info()

In [ ]:
try:
    from pytabkit import TabM_D_Classifier, RealMLP_TD_Classifier
except:
    !pip install -q pytabkit
    from pytabkit import TabM_D_Classifier, RealMLP_TD_Classifier

try:
    import xgboost as xgb
except:
    !pip install -q xgboost
    import xgboost as xgb

In [ ]:
## -- Data Manipulation --
import numpy as np, pandas as pd, random

## -- Visualization --
from IPython.display import display, Image
import matplotlib.pyplot as plt
import seaborn as sns

## -- Functional Tools --
import itertools
from tqdm import tqdm
from time import time, sleep
from importlib.metadata import version

## -- Machine Learning --
from sklearn.calibration import CalibrationDisplay
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, KBinsDiscretizer, TargetEncoder as skTE

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

import warnings
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore')

## -- Set Configuration --
# sklearn.set_config(transform_output="pandas")
# pd.options.mode.copy_on_write = True
pd.set_option("display.max_columns", 1000)
sns.set_style("whitegrid")
# plt.style.use('ggplot')

print("xgboost  version:", xgb.__version__)
print("PyTabKit version:", version("pytabkit"))

In [ ]:
TEAL, GOLD, CORAL, BLUE, DARK, CRIM, MUTED, INK, PAPER = '#168c91','#e8b44f','#dc705a',\
                                                         'dodgerblue','#0B4F7A','crimson',\
                                                         '#627D98','#102A43','#faf9f5'
plt.rcParams.update({
    'figure.facecolor': PAPER,
    'axes.facecolor': PAPER,
    'savefig.facecolor': PAPER,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 13,
    'axes.titleweight': 'semibold',
    'axes.labelcolor': INK,
    'text.color': INK,
    'xtick.color': DARK,
    'ytick.color': DARK,
    'axes.edgecolor': MUTED,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.color': MUTED,
    'patch.edgecolor': DARK,
    # 'patch.force_edgecolor': True,
    # 'patch.linewidth': 1.5,
    # 'figure.dpi':110,
    # 'savefig.dpi':160,
})
# CMAP = LinearSegmentedColormap.from_list('atlas', ['#f0f5ef','#a9d6ca',TEAL,INK])
sns.set_palette([TEAL, GOLD, CORAL, BLUE, MUTED, CRIM, DARK, INK, PAPER])

## -- Set Global Seed --
CFG = {
    'FOLDS':  5,
    'SEED':   42,
    'GREEN':  '\033[32m',
    'YELLOW': '\033[33m',
    'RED':    '\033[31m',
    'RESET':  '\033[0m'
}

print(f"CLASSIC {CFG['GREEN']} GREEN {CFG['RESET']} {CFG['YELLOW']} YELLOW {CFG['RESET']}")

In [ ]:
## -- ⚠️ IMPORTANT: SELECT PLATFORM ⚠️ --
PLATFORM = "kaggle" # -> 'colab' 'kaggle'

if PLATFORM == "kaggle":
    PATH = "/kaggle/input/competitions/playground-series-s6e10/"
    submit = pd.read_csv(PATH+"sample_submission.csv")
    train = pd.read_csv(PATH+"train.csv")
    test = pd.read_csv(PATH+"test.csv")

    ORIG_PATH = "/kaggle/input/datasets/arseniyshutko/binary-aviation-satisfaction-129k/"
    orig = pd.read_csv(ORIG_PATH+"data.csv")
elif PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")

    PATH   = "/content/drive/MyDrive/-- shared_notebooks --/Ps6e10 | Airline Satisfaction/_airline_data/"
    submit = pd.read_csv(PATH+"sample_submission.csv")
    train  = pd.read_csv(PATH+"train.csv")
    test   = pd.read_csv(PATH+"test.csv")
    orig = pd.read_csv(PATH+"data.csv")

# if torch.cuda.is_available():
#     submit = cudf.DataFrame(submit)
#     train = cudf.DataFrame(train)
#     test = cudf.DataFrame(test)
#     orig = cudf.DataFrame(orig)

## =================================================================================

for (name, df) in {"Train": train, "Test": test, 'Orig': orig}.items(): #
    print(f"{name} shape: {df.shape}")

print(f"\nFrame type: {type(train)}")

## FEATURE ENGINEERING

In [ ]:
# ## -- Cyclic encoding --
# CYCLICS = []
# cy_cols = [c for c in train.columns if c.endswith("_hours")]

# for col in tqdm(cy_cols):
#     for p in [12, 24]:
#         n_s = f"{col}_sin_{p}"

#         train[n_s] = np.sin(2 * np.pi * train[col] / p).astype('float32')
#         orig[n_s]  = np.sin(2 * np.pi * orig[col] / p).astype('float32')
#         test[n_s]  = np.sin(2 * np.pi * test[col] / p).astype('float32')
#         CYCLICS.append(n_s)

#         # n_c = f"{col}_cos_{p}"
#         # train[n_c] = np.cos(2 * np.pi * train[col] / p).astype('float32')
#         # test[n_c] = np.cos(2 * np.pi * test[col] / p).astype('float32')
#         # orig[n_c] = np.cos(2 * np.pi * orig[col] / p).astype('float32')
#         # CYCLICS.append(n_c)

# # NUMS.extend(CYCLICS)

# print(f"Cylcic Features created: {len(CYCLICS)}")
# print(train[CYCLICS].describe().T)

# train[CYCLICS].head()

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin ## ===== Target/CategoryMean ENCODERS =====

class customTE(BaseEstimator, TransformerMixin):
    """
    Target Encoder that supports multiple aggregation functions,
    internal cross-validation for leakage prevention, and smoothing.

    Parameters
    ----------
    cols_to_encode : list of str
        List of column names to be target encoded.

    aggs : list of str, default=['mean']
        List of aggregation functions to apply. Any function accepted by
        pandas' `.agg()` method is supported, such as:
        'mean', 'std', 'var', 'min', 'max', 'skew', 'nunique',
        'count', 'sum', 'median'.
        Smoothing is applied only to the 'mean' aggregation.

    cv : int, default=5
        Number of folds for cross-validation in fit_transform.

    smooth : float or 'auto', default='auto'
        The smoothing parameter `m`. A larger value puts more weight on the
        global mean. If 'auto', an empirical Bayes estimate is used.

    drop_original : bool, default=False
        If True, the original columns to be encoded are dropped.
    """
    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=False):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        """
        Learn mappings from the entire dataset.
        These mappings are used for the transform method on validation/test data.
        """
        temp_df = X.copy()
        temp_df['target'] = y

        # Learn global statistics for each aggregation
        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        # Learn category-specific mappings
        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)
                self.mappings_[col][agg_func] = mapping

        return self

    def transform(self, X):
        """
        Apply learned mappings to the data.
        Unseen categories are filled with global statistics.
        """
        X_transformed = X.copy()
        for col in tqdm(self.cols_to_encode, desc="TE encoding"):
            for agg_func in self.aggs:
                new_col_name = f"TE_{col}_{agg_func}"
                map_series = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(map_series)
                X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

    def fit_transform(self, X, y):
        """
        Fit and transform the data using internal cross-validation to prevent leakage.
        """
        # First, fit on the entire dataset to get global mappings for transform method
        self.fit(X, y)

        # Initialize an empty DataFrame to store encoded features
        encoded_features = pd.DataFrame(index=X.index)

        kf = KFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]

            temp_df_train = X_train.copy()
            temp_df_train['target'] = y_train

            for col in self.cols_to_encode:
                # --- Calculate mappings only on the training part of the fold ---
                for agg_func in self.aggs:
                    new_col_name = f"TE_{col}_{agg_func}"

                    # Calculate global stat for this fold
                    fold_global_stat = y_train.agg(agg_func)

                    # Calculate category stats for this fold
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    # --- Apply smoothing only for 'mean' aggregation ---
                    if agg_func == 'mean':
                        counts = temp_df_train.groupby(col)['target'].count()

                        m = self.smooth
                        if self.smooth == 'auto':
                            # Empirical Bayes smoothing
                            variance_between = mapping.var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].var().mean()
                            if variance_between > 0:
                                m = avg_variance_within / variance_between
                            else:
                                m = 0  # No smoothing if no variance between groups

                        # Apply smoothing formula
                        smoothed_mapping = (counts * mapping + m * fold_global_stat) / (counts + m)
                        encoded_values = X_val[col].map(smoothed_mapping)
                    else:
                        encoded_values = X_val[col].map(mapping)

                    # Store encoded values for the validation fold
                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(fold_global_stat)

        # Merge with original DataFrame
        X_transformed = X.copy()
        for col in encoded_features.columns:
            X_transformed[col] = encoded_features[col]

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed


class CategoryMeanTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols=None):
        self.cat_cols = cat_cols
        self.mappings_ = {}
    def fit(self, X, y):
        X = X.copy()
        if self.cat_cols is None:
            self.cat_cols = X.select_dtypes(include=['category']).columns.tolist()
        self.mappings_ = {}
        for col in self.cat_cols:
            df_temp = pd.DataFrame({col: X[col], 'y': y})
            group_means = df_temp.groupby(col, dropna=False)['y'].mean()
            sorted_categories = group_means.sort_values().index
            self.mappings_[col] = {cat: i for i, cat in enumerate(sorted_categories)}
        return self

    def transform(self, X, y=None):
        X = X.copy()
        for col, mapping in self.mappings_.items():
            if col in X.columns:
                X[col] = X[col].map(mapping)
        return X

In [ ]:
def original_TE(
    orig: pd.DataFrame,
    df: pd.DataFrame,
    features: list=None,
    target: str=None,
    aggs: list=None,
    fill_nan: bool=False,
):

    if features is None or len(features) == 0:
        return df.copy()

    if aggs is None or len(aggs) == 0:
        return df.copy()

    X_df = df.copy()

    orig_cols = []
    maps = {}

    valid_features = [col for col in features if col in orig.columns]

    pbar = tqdm(valid_features)
    for col in pbar:
        for agg_ in aggs:
            agg_key = agg_.lower()
            new_col = f"oTE_{col}_{agg_key}"
            pbar.set_description(f"Augmenting {col}")
            map_key = (col, agg_key)
            if map_key not in maps:
                try:
                    if agg_key == 'mean':
                        map_df = (orig.groupby(col)[target].mean().reset_index(name=new_col))
                    elif agg_key == 'median':
                        map_df = (orig.groupby(col)[target].median().reset_index(name=new_col))
                    elif agg_key == 'count':
                        map_df = (orig.groupby(col).size().reset_index(name=new_col)) #.astype('int32')
                    elif agg_key == 'nunique':
                        map_df = (orig.groupby(col)[target].nunique().reset_index(name=new_col)) #.astype('int32')
                    elif agg_key == 'std':
                        map_df = (orig.groupby(col)[target].std().reset_index(name=new_col))
                    elif agg_key == 'skew':
                        map_df = (orig.groupby(col)[target].skew().reset_index(name=new_col))
                    elif agg_key == 'max':
                        map_df = (orig.groupby(col)[target].max().reset_index(name=new_col))
                    elif agg_key == 'min':
                        map_df = (orig.groupby(col)[target].min().reset_index(name=new_col))
                    else:
                        continue
                except Exception as e:
                    print(f"Warning: failed to create map for col={col}, agg={agg_}: {e}")
                    continue

                maps[map_key] = map_df

            map_df = maps.get(map_key)
            if map_df is None:
                continue

            # Merge maps into each split
            X_df = X_df.merge(map_df, on=col, how='left')

            orig_cols.append(new_col)

        pbar.set_description(f"Original augment complete!")

    global_mean   = orig[target].mean()
    global_median = orig[target].median()

    def fill_conditionally(df):
        for c in orig_cols:
            if c.endswith(('_mean', '_max', '_min')):
                df[c] = df[c].fillna(global_mean)
            elif c.endswith('_median'):
                df[c] = df[c].fillna(global_median)
            else:
                df[c] = df[c].fillna(0)
        return df

    if fill_nan:
        X_df = fill_conditionally(X_df)

    return X_df, orig_cols

print("-- Data augment function ready! --")

In [ ]:
# FEATURES = [c for c in train.columns if c not in [ID, TARGET]]
# print("Total Features:", len(FEATURES))

# combined = pd.concat([train, test, orig])
# ## ------------------------------------------------------------
# # # -- Duplicate Nums as Cats --
# # CATS_2 = []
# # for c in tqdm(NUMS, title="nums_to_cats"):
# #     n = f"{c}_cat"
# #     combined[n] = combined[c].copy()
# #     CATS_2.append(n)
# # FEATURES.extend(CATS_2)
# ## ------------------------------------------------------------
# ## -- Factorize CATS --
# for c in tqdm(CATS, desc="factorize_cats"): # +INTER + new_cats
#     combined[c] = combined[c].factorize()[0]

# for c in tqdm(CATS, desc="categorize_cats"):
#     combined[c] = combined[c].astype("category")

# train = combined.iloc[:len(train)][FEATURES+[TARGET]]
# test  = combined.iloc[len(train):len(train)+len(test)][FEATURES]
# orig  = combined.iloc[len(train)+len(test):]

# del combined

In [ ]:
ID = 'id'
TARGET = 'satisfaction'
train[TARGET] = train[TARGET].map({False: 0, True: 1})
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
del train, test
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
important_combos = [
    ('Class', 'Type of Travel', 'Gender'),
    ('Class', 'Type of Travel', 'Age'),
    ('Flight Distance', 'Departure/Arrival time convenient'),
]
def feature_engineering(df, fit=False):
    # Fill NaNs
    for col in cat_cols:
        df[col] = df[col].fillna("missing")
    for col in num_cols:
        df[col] = df[col].fillna(0.0)
    
    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Count encoding
    for col in cat_cols + ['Leg room service']:
        count_name = f"_{col}_count"
        if fit:
            count_map = df[col].value_counts()
            category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]
        df[count_name] = df[col].map(count_map).fillna(0).astype('int32')

    # Create interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category') 

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape, "\n")

In [ ]:
X.dtypes

In [ ]:
X.nunique()

# ML TRAINING

In [ ]:
real_params = {
    'random_state': 42,
    'verbosity': 2,
    'val_metric_name': '1-auc_ovr',

    'n_ens': 8,
    'n_epochs': 3,
    'batch_size': 256,
    'use_early_stopping': False,
    'early_stopping_additive_patience': 10,
    'early_stopping_multiplicative_patience': 1,

    'lr': 0.053,
    'wd': 0.015,
    'sq_mom': 0.988,
    'lr_sched': 'flat_anneal',
    'wd_sched': 'cos_log_15',
    'first_layer_lr_factor': 0.25,

    'embedding_size': 5,
    'max_one_hot_cat_size': 1,
    'hidden_sizes': [512, 256, 128],
    'act': 'silu',
    'p_drop': 0.05,
    'p_drop_sched': 'pow5',

    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_act_name': 'gelu',
    'plr_lr_factor': 0.1151,
    'plr_sigma': 2.33,

    'ls_eps': 0.01,
    'ls_eps_sched': 'sqrt_cos',

    'add_front_scale': False,
    'bias_init_mode': 'neg-uniform-dynamic-2',
    'tfms': ['one_hot', 'median_center', 'robust_scale',
             'smooth_clip', 'embedding', 'l2_normalize'],
}

tabm_params = {
    'random_state': 42,
    'verbosity': 2,
    'val_metric_name': '1-auc_ovr',
    
    'arch_type': 'tabm-normal',
    'num_emb_type': 'pwl',
    # 'allow_amp': True,
    'tabm_k': 24,

    'lr': 0.002,
    'n_epochs': 2,
    'patience': 2,

    'batch_size': 256,
    'd_block': 512,
    # 'n_blocks': 3,
    
    'd_embedding': 6,
    # 'dropout': 0.05,
    # 'num_emb_n_bins': 48,

    'compile_model': False,
    'gradient_clipping_norm': None,
    'tfms': ['quantile_tabr'],
    'weight_decay': 0.0,
}

nn_params = {
    "tabm": tabm_params,
    "realmlp": real_params,
}

nn_params["tabm"]

In [ ]:
LR = 0.01

gbt_params = {
    # ------------------------------------
    "objective": "binary:logistic",
    "eval_metric": "auc",
    # "enable_categorical": True,
    "learning_rate": LR,
    "n_estimators": 10_000,
    "max_depth": 6,
    "subsample": 0.9,
    "colsample_bytree": 0.6,
    "reg_alpha": 0.1, # 1.0, 3.0,
    "reg_lambda": 5.0, # 5.0, 10.0,
    "min_child_weight": 10, # 10, 50,
    # ------------------------------------
    "max_bin": 512,
    # "scale_pos_weight": sum(train[TARGET] == 0) / sum(train[TARGET] == 1),
    "random_state": 42,
    "verbosity": 0,
    "n_jobs": -1,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

loss_params = {
    "grow_policy": "lossguide",
    # ----------------------------------
    # "max_leaves": 64,
    "max_depth": 0, # 10, 0
    # ----------------------------------
    "objective": "binary:logistic",
    "eval_metric": "auc",
    # "enable_categorical": True,
    "learning_rate": LR,
    "n_estimators": 10_000,
    "subsample": 0.9,
    "colsample_bytree": 0.6,
    # "reg_alpha": 0.1, # 0.5, 5.0
    "reg_lambda": 0.1, # 5.0, 10.0
    "min_child_weight": 5, # 20, 40
    # "max_bin": 512,
    # "scale_pos_weight": sum(train[TARGET] == 0) / sum(train[TARGET] == 1),
    "random_state": 42,
    "verbosity": 0,
    "n_jobs": -1,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
} 

grad_params = {
    'sampling_method': "gradient_based",
    # ------------------------------------
    "objective": "binary:logistic",
    "eval_metric": "auc",
    # "enable_categorical": True,
    "learning_rate": LR,
    "n_estimators": 10_000,
    "max_depth": 10,
    "subsample": 0.9,
    "colsample_bytree": 0.4,
    # "reg_alpha": 0.0, # 5.0
    "reg_lambda": 0.001, # 10.0
    "min_child_weight": 5, # 40
    # "max_bin": 512,
    # "scale_pos_weight": sum(train[TARGET] == 0) / sum(train[TARGET] == 1),
    "random_state": 42,
    "verbosity": 0,
    "n_jobs": -1,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

m2_params = [
    ("XGBgbt", gbt_params),
    ("XGBlos", loss_params),
    ("XGBgra", grad_params),
]

m2_params[0]

In [ ]:
X.columns == X_test.columns

In [ ]:
all_predictions = {}

FOLDS = 5
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

USE_FULL_TRAIN = True
x_sample, x_sample2 = train_test_split(
    pd.concat([X, y], axis=1),
    train_size=300_000,
    stratify=y,
    random_state=42,
)
train_X = pd.concat([X, y], axis=1).copy() if USE_FULL_TRAIN else x_sample.reset_index(drop=True)

print(train_X.shape, type(train_X))
train_X.head()

In [ ]:
FEATURES = [c for c in train_X.columns if c not in [ID, TARGET]]
FEATURES = list(set(FEATURES))

print("Total Features:", len(FEATURES))

# train_X

In [ ]:
## -- Pseudo Trainer --

_id = 1 # 0->"gbt", 1->"loss", 2->"grad"
m2_name = m2_params[_id][0] 

USE_ORIGINAL = False
USE_TABM     = True if _id == 2 else False
USE_TE       = True
DROP_TE      = True

m1_name = "TabM" if USE_TABM else "RealMLP"

# Handle optional original table aggregations securely if flags enable them
if USE_ORIGINAL and "original_TE" in globals():
    feats_update = FEATURES.copy()
    aggs_ = ["mean", "count"]
    X_tr, orig_cols = original_TE(
        orig, train_X, target=TARGET, features=base_cols, aggs=aggs_, fill_nan=True
    )
    x_test, _ = original_TE(
        orig, X_test, target=TARGET, features=base_cols, aggs=aggs_, fill_nan=True
    )
    feats_update.extend(orig_cols)
    X_1 = X_tr[feats_update].copy()
    y_1 = X_tr[TARGET].astype("int8")
    X_test = x_test[feats_update].copy()
else:
    X_1 = train_X[FEATURES].copy()
    y_1 = train_X[TARGET].astype("int8")
    X_tst = X_test[FEATURES].copy()

oof_preds  = np.zeros(len(X_1))
test_preds = np.zeros(len(X_tst))

fold_scores = []
m1_scores   = []
feat_importances = []

# all_cats = cat_cols #+ new_cats
# all_nums = [c for c in FEATURES if c not in all_cats]

t0 = time()

## -- Initiate Training --
for idx, (train_idx, val_idx) in enumerate(skf.split(X_1, y_1), 1):
    print(f"\n{'#'*30}")
    print(f"+++ FOLD {idx}/{skf.n_splits} | ⚡️ {m2_name}/{m1_name}")
    print(f"{'#'*30}")

    X_train, X_valid = X_1.iloc[train_idx], X_1.iloc[val_idx]
    y_train, y_valid = y_1.iloc[train_idx], y_1.iloc[val_idx]
    X_test_fold = X_tst.copy()

    # ## -- OPTION A: Concatenate original data --
    # X_train = pd.concat([X_train, orig[FEATURES]], ignore_index=True)
    # y_train = pd.concat([y_train, orig[TARGET]], ignore_index=True)
    # y_train = np.concatenate([y_train, orig[TARGET].values], axis=0)

    if USE_TE:
        te_cols = combo_names
        # custom TE
        # print(f" ⧗ custom TE ({len(te_cols)}) features", end=" | ")
        # te_enc  = customTE(te_cols, cv=5, smooth='auto', aggs=['mean'], drop_original=DROP_TE)
        # X_train = te_enc.fit_transform(X_train, y_train)
        # X_valid = te_enc.transform(X_valid)
        # X_test_fold = te_enc.transform(X_test_fold)

        # sklearn TE
        print(f" ⧗ sklearn TE ({len(te_cols)}) features", end=" | ")
        te_enc = skTE(target_type='binary', cv=5, smooth='auto', random_state=42)
        tr_enc = te_enc.fit_transform(X_train[te_cols], y_train)
        va_enc = te_enc.transform(X_valid[te_cols])
        ts_enc = te_enc.transform(X_test_fold[te_cols])

        te_names = [f"TE_{col}_" for col in te_cols]
        X_train[te_names]  = tr_enc
        X_valid[te_names] = va_enc
        X_test_fold[te_names] = ts_enc

        combos_to_drop = ['Class_Type of Travel_Age_', 'Flight Distance_Departure/Arrival time convenient_']
        X_train = X_train.drop(columns=combos_to_drop)
        X_valid = X_valid.drop(columns=combos_to_drop)
        X_test_fold = X_test_fold.drop(columns=combos_to_drop)

    # # ---- Imputing ----
    # # imputer = IterativeImputer(max_iter=20, tol=1e-3, random_state=0) # n_nearest_features=10, 
    # imputer = SimpleImputer(strategy="median")
    # X_train[all_nums] = imputer.fit_transform(X_train[all_nums], y_train)
    # X_valid[all_nums] = imputer.transform(X_valid[all_nums])
    # X_test[all_nums]  = imputer.transform(X_test[all_nums])

    # cme_encoder = CategoryMeanTransformer(cat_cols=new_cats)
    # X_train = cme_encoder.fit_transform(X_train, y_train)
    # X_valid = cme_encoder.transform(X_valid)
    # X_test  = cme_encoder.transform(X_test)

    all_cats = [c for c in cat_cols if c not in combos_to_drop]

    ## -- Select base model --
    if USE_TABM:
        model_1 = TabM_D_Classifier(**nn_params["tabm"])
    else:
        model_1 = RealMLP_TD_Classifier(**nn_params["realmlp"])

    model_1.fit(X_train, y_train, X_valid, y_valid) #, cat_col_names=all_cats
    oof_m1  = model_1.predict_proba(X_valid)[:, 1]
    test_m1 = model_1.predict_proba(X_test_fold)[:, 1]

    score_m1 = roc_auc_score(y_valid, oof_m1)
    print(f"{CFG['YELLOW']} • {m1_name} AUC: {score_m1:.6f}{CFG['RESET']}")
    m1_scores.append(score_m1)

    # ## -- Target encode 'nums_cats' prior xgb pseudo
    # if USE_TE:
    #     te_cols = [c for c in all_cats if c.endswith('_cat_')]
    #     # custom TE
    #     # print(f" ⧗ custom TE ({len(te_cols)}) features", end=" | ")
    #     # te_enc  = customTE(te_cols, cv=5, smooth='auto', aggs=['mean'], drop_original=DROP_TE)
    #     # X_train = te_enc.fit_transform(X_train, y_train)
    #     # X_valid = te_enc.transform(X_valid)
    #     # X_test_fold = te_enc.transform(X_test_fold)

    #     # sklearn TE
    #     print(f" ⧗ sklearn TE ({len(te_cols)}) features")
    #     te_enc = skTE(target_type='binary', cv=5, smooth='auto', random_state=42)
    #     tr_enc = te_enc.fit_transform(X_train[te_cols], y_train)
    #     va_enc = te_enc.transform(X_valid[te_cols])
    #     ts_enc = te_enc.transform(X_test_fold[te_cols])

    #     te_names = [f"_{col}TE" for col in te_cols]
    #     X_train[te_names] = tr_enc
    #     X_valid[te_names] = va_enc
    #     X_test_fold[te_names] = ts_enc

    #     X_train = X_train.drop(columns=te_cols)
    #     X_valid = X_valid.drop(columns=te_cols)
    #     X_test_fold = X_test_fold.drop(columns=te_cols)
        
    print(f"train= {X_train.shape}, valid= {X_valid.shape}")
    if idx == 1:
        display(X_train.head(3))

    ## -- Pseudo labels from model_1 as features for model_2 --
    X_train_full = pd.concat([X_train, X_valid, X_test_fold])
    y_train_full = np.concatenate([y_train.to_numpy(), oof_m1, test_m1], axis=0)

    for c in cat_cols[:4]:
        X_train_full[c] = X_train_full[c].astype('category')

    X_valid_2 = X_train_full.iloc[len(X_train):len(X_train)+len(X_valid)]
    X_test_2  = X_train_full.iloc[len(X_train)+len(X_valid):]

    dtrain = xgb.DMatrix(X_train_full, y_train_full, enable_categorical=True)
    dval   = xgb.DMatrix(X_valid_2, y_valid, enable_categorical=True)
    dtest  = xgb.DMatrix(X_test_2, enable_categorical=True)

    model_2 = xgb.train(
        m2_params[_id][1],
        dtrain,
        num_boost_round = 10_000,
        evals = [(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds = 300,
        verbose_eval = 500,
    )
    
    imp_dict = model_2.get_score(importance_type='gain')
    imp_values = [imp_dict.get(feat, 0.0) for feat in X_train_full.columns]
    feat_importances.append(np.asarray(imp_values))

    oof_preds[val_idx] = model_2.predict(dval, iteration_range=(0, model_2.best_iteration+1))
    score_m2 = roc_auc_score(y_valid, oof_preds[val_idx])
    fold_scores.append(score_m2)

    print(f"{CFG['YELLOW']} • {m2_name} AUC: {score_m2:.6f}{CFG['RESET']}")

    if score_m2 >= score_m1:
        print(f"{CFG['GREEN']} • 🍀 improved: {score_m2-score_m1:.6f}{CFG['RESET']}")
    else:
        print(f"{CFG['RED']} • ⚠️ declined: {score_m1-score_m2:.6f}{CFG['RESET']}")

    test_preds += model_2.predict(dtest, iteration_range=(0, model_2.best_iteration+1)) / FOLDS

    # gc.collect()
    torch.cuda.empty_cache()

print('\n'+'='*50)
print(f"{FOLDS}-FOLD CV: {m2_name}/{m1_name} | shape: {X_train_full.shape}")
print('='*50)
for i, (s1, s2) in enumerate(zip(m1_scores, fold_scores), 1):  
    print(f" • Fold {i} AUC: {m1_name}={s1:.6f} | {m2_name}={s2:.6f}")  

oof_auc = np.round(roc_auc_score(y_1, oof_preds), 6)

avg_auc = np.round(np.mean(fold_scores), 6)
std_auc = np.round(np.std(fold_scores), 6)

m1avg_auc = np.round(np.mean(m1_scores), 6)
m1std_auc = np.round(np.std(m1_scores), 6)

print(f"{'='*50}")
print(f"| PSUEDO OOF AUC: {oof_auc:>5}")
print(f"| PSUEDO AVG AUC: {avg_auc:>5} ± {std_auc}")
print(f"| {m1_name:<15} {m1avg_auc} ± {m1std_auc}")
print(f"{'='*50}")

print()
print(f"Training time: {(time()-t0)/60:.2f} mins {'+'*24}")

In [ ]:
# ==================================================
# 5-FOLD CV: XGBgra/TabM | shape: (999479, 47)
# ==================================================
#  • Fold 1 AUC: TabM=0.960657 | XGBgra=0.960667
#  • Fold 2 AUC: TabM=0.959420 | XGBgra=0.959593
#  • Fold 3 AUC: TabM=0.961104 | XGBgra=0.961241
#  • Fold 4 AUC: TabM=0.960472 | XGBgra=0.960517
#  • Fold 5 AUC: TabM=0.960401 | XGBgra=0.960497
# ==================================================
# | PSUEDO OOF AUC: 0.96048
# | PSUEDO AVG AUC: 0.960503 ± 0.000529
# | TabM            0.960411 ± 0.000553
# ==================================================

# Training time: 23.36 mins ++++++++++++++++++++++++

In [ ]:
print(f"{'='*50}")
print(f"| PSUEDO OOF AUC: {oof_auc:>5}")
print(f"| PSUEDO AVG AUC: {avg_auc:>5} ± {std_auc}")
print(f"| {m1_name:<15} {m1avg_auc} ± {m1std_auc}")
print(f"{'='*50}")

print()
print(f"Training time: {(time()-t0)/60:.2f} mins {'+'*24}")

In [ ]:
## -- Plot predictions errors --
name = f"PSEUDO_{m2_name}_{m1_name}_" + str(oof_auc).split('.')[1]

_, axs = plt.subplots(1, 3, figsize=(16, 5))

CalibrationDisplay.from_predictions(y_1, oof_preds, n_bins=10, name=name, ax=axs[0])
axs[0].set_title("Calibrated predictions")

RocCurveDisplay.from_predictions(y_1, oof_preds, name=name, ax=axs[1])
axs[1].set_title("ROC Curve")
# axs[1].grid(False)

sns.histplot(oof_preds, ax=axs[2])
axs[2].set_title("Distribution")

plt.suptitle(name, fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
## -- Get feature importances --
top_n = 40
    
fi_df = pd.DataFrame(
    {
        "feature": X_train_full.columns,
        "importance_score": np.mean(feat_importances, axis=0),
    }
).sort_values(by="importance_score", ascending=False).reset_index(drop=True)

plt.figure(figsize=(16, top_n*0.3))
sns.barplot(data=fi_df.head(top_n), x="importance_score", y="feature", palette='plasma')
plt.title(f"Avg. {FOLDS}-Fold feature importances: {name}",size=15,weight="semibold",loc='left')

plt.tight_layout()
plt.show()

print()

## SUBMISSION

In [ ]:
## -- Save submission file --
submit[TARGET] = test_preds
submit.to_csv(f"submit_{name}.csv", index=False)

## -- Save oof/test predictions --
np.save(f"oof_{name}.npy", oof_preds)
np.save(f"test_{name}.npy", test_preds)

sns.histplot(submit[TARGET], bins=50)
plt.show()

submit

In [ ]:
# !rm -r /kaggle/working